# PFE Navigator — Système RAG

## 1. Présentation du projet

PFE Navigator est un système intelligent basé sur
la génération augmentée par récupération (RAG).

Il permet aux étudiants de poser des questions concernant
leur Projet de Fin d'Études (PFE) et d'obtenir des réponses
à partir d'un ensemble de documents PDF.

## 2. Objectifs

- Exploiter un corpus de cinq documents PDF.
- Tester un LLM seul, sans contexte documentaire.
- Construire un système de recherche documentaire.
- Générer des réponses fondées sur les passages récupérés.
- Afficher les sources utilisées.
- Comparer les résultats du LLM seul et du RAG.
- Identifier et corriger un échec du système.

 ## Étape 1 — Importer les bibliothèques

In [8]:
import os
import re
import json
import numpy as np
import ollama

from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

### 1.1  Configuration Ollama

In [9]:
# Configuration du modèle local
MODEL = "qwen2.5:7b"

# Vérification du modèle Ollama
modeles = ollama.list()
noms_modeles = [modele.model for modele in modeles.models]

if not any(nom.startswith(MODEL) for nom in noms_modeles):
    raise ValueError("Le modèle Qwen 2.5 7B n'est pas installé.")

print("Ollama est connecté avec succès.")
print("Modèle utilisé :", MODEL)

# Gestion des chemins du projet
if Path("documents").exists():
    DOSSIER_DOCUMENTS = Path("documents")
    DOSSIER_RESULTATS = Path("results")
elif Path("../documents").exists():
    DOSSIER_DOCUMENTS = Path("../documents")
    DOSSIER_RESULTATS = Path("../results")
else:
    raise FileNotFoundError("Dossier documents introuvable.")

DOSSIER_RESULTATS.mkdir(exist_ok=True)

Ollama est connecté avec succès.
Modèle utilisé : qwen2.5:7b


### 1.2  Définir les questions de test

In [10]:
questions = [
    "Quelle est la date limite de dépôt du rapport final de PFE, et sous quelle forme faut-il le remettre ?",

    "Comment est calculée la note finale du PFE, et à partir de quelle note est-il validé ?",

    "Quelle police et quel interligne faut-il utiliser dans le rapport ?",

    "Quelle est la date de la session de rattrapage des soutenances ?",

    "Quel est le montant de la bourse accordée aux étudiants effectuant leur PFE ?"
]

for i, question in enumerate(questions, start=1):
    print(f"Question {i} : {question}")

Question 1 : Quelle est la date limite de dépôt du rapport final de PFE, et sous quelle forme faut-il le remettre ?
Question 2 : Comment est calculée la note finale du PFE, et à partir de quelle note est-il validé ?
Question 3 : Quelle police et quel interligne faut-il utiliser dans le rapport ?
Question 4 : Quelle est la date de la session de rattrapage des soutenances ?
Question 5 : Quel est le montant de la bourse accordée aux étudiants effectuant leur PFE ?



## 2. Test du LLM seul

Dans cette étape, nous évaluons les capacités du modèle Qwen 2.5 7B à répondre directement aux questions des étudiants, sans lui fournir les documents de l'ISIT.

Ce test constitue une référence pour la comparaison avec le système RAG.

Nous observons notamment la pertinence des réponses et la capacité du modèle à reconnaître les informations qu'il ne connaît pas.



## . Intégration d'un LLM local avec Ollama

Afin de ne plus dépendre d'une API externe et de ses quotas, nous remplaçons Gemini par Ollama, qui permet d'exécuter un modèle de langage localement.

Nous utilisons Qwen 2.5 7B pour générer les réponses à partir des passages documentaires récupérés par notre système RAG.


In [11]:

import ollama

MODEL = "qwen2.5:7b"

# Vérification de la disponibilité du modèle
modeles = ollama.list()

noms_modeles = [
    modele.model for modele in modeles.models
]

if MODEL not in noms_modeles:
    raise ValueError("Le modèle Qwen 2.5 7B n'est pas installé.")

print("Ollama est connecté avec succès.")
print("Modèle utilisé :", MODEL)


Ollama est connecté avec succès.
Modèle utilisé : qwen2.5:7b


test des cinq questions

In [12]:
resultats_llm = []

for i, question in enumerate(questions, start=1):

    print("\n" + "=" * 60)
    print(f"QUESTION {i} : {question}")
    print("=" * 60)

    try:
        response = ollama.chat(
            model=MODEL,
            messages=[
                {
                    "role": "user",
                    "content": question
                }
            ],
            options={
                "temperature": 0.2,
                "num_predict": 300
            }
        )

        reponse = response.message.content

        statut = "success"
        erreur = None

        print("\nRéponse du LLM :")
        print(reponse)

    except Exception as e:
        reponse = None
        statut = "error"
        erreur = str(e)

        print("Erreur :", erreur)

    resultat = {
        "question": question,
        "reponse": reponse,
        "statut": statut,
        "erreur": erreur
    }

    resultats_llm.append(resultat)

# Sauvegarde des résultats
with open(
    DOSSIER_RESULTATS / "llm_sans_rag.json",
    "w",
    encoding="utf-8"
) as fichier:
    json.dump(
        resultats_llm,
        fichier,
        ensure_ascii=False,
        indent=4
    )

print("\nTest LLM seul terminé.")



QUESTION 1 : Quelle est la date limite de dépôt du rapport final de PFE, et sous quelle forme faut-il le remettre ?

Réponse du LLM :
Pour répondre précisément à votre question, il serait utile de connaître les détails spécifiques de votre programme de PFE (Projet Final d'Études) ou de votre institution d'enseignement. Cependant, je peux vous donner quelques conseils généraux sur la date limite et la forme de remise du rapport final :

1. Date limite de dépôt :
   - Elle est généralement fixée par votre université ou l'organisme responsable du PFE.
   - Elle peut varier selon les filières et les établissements.
   - Elle est souvent annoncée au début de l'année académique ou à la fin du semestre.

2. Forme de remise du rapport :
   - Version papier : Certains établissements préfèrent une version imprimée.
   - Version électronique : De plus en plus d'établissements acceptent des versions numériques.
   - Version PDF : C'est souvent la forme recommandée, car elle est facile à transmett

## étape 3 - Préparer les documents

### 3.1 Charger et extraire le contenu des PDF


Préparation des documents

Nous utilisons cinq documents PDF relatifs au Projet de Fin d'Études.

La préparation comprend trois opérations :
1. Extraction du texte à partir des fichiers PDF.
2. Nettoyage du texte extrait.
3. Découpage du texte en petits passages appelés chunks.

Nous commençons avec une taille de 400 mots et un chevauchement de 50 mots afin de conserver le contexte entre les passages.


In [13]:
# Recherche des fichiers PDF
fichiers_pdf = sorted(DOSSIER_DOCUMENTS.glob("*.pdf"))

if not fichiers_pdf:
    raise FileNotFoundError("Aucun PDF trouvé.")

pages_data = []

# Extraction du texte
for fichier in fichiers_pdf:

    lecteur = PdfReader(str(fichier))

    for numero_page, page in enumerate(lecteur.pages, start=1):

        texte = page.extract_text() or ""
        texte = re.sub(r"\s+", " ", texte).strip()

        if texte:
            pages_data.append({
                "source": fichier.name,
                "page": numero_page,
                "texte": texte
            })

print("Nombre de PDF :", len(fichiers_pdf))
print("Nombre de pages extraites :", len(pages_data))

# Fonction de découpage en chunks
def creer_chunks(pages, taille, overlap):

    resultat = []
    documents = {}

    for page in pages:
        documents.setdefault(page["source"], []).append(page)

    pas = taille - overlap

    for nom_document, liste_pages in documents.items():

        mots_document = []
        pages_mots = []

        for page in liste_pages:

            mots_page = page["texte"].split()
            mots_document.extend(mots_page)
            pages_mots.extend([page["page"]] * len(mots_page))

        for debut in range(0, len(mots_document), pas):

            fin = min(debut + taille, len(mots_document))

            texte_chunk = " ".join(mots_document[debut:fin])

            if texte_chunk.strip():

                resultat.append({
                    "texte": texte_chunk,
                    "source": nom_document,
                    "page_debut": pages_mots[debut],
                    "page_fin": pages_mots[fin - 1],
                    "nombre_mots": fin - debut
                })

            if fin == len(mots_document):
                break

    return resultat


# Création des chunks initiaux
taille_chunk = 400
overlap = 50

chunks = creer_chunks(
    pages_data,
    taille_chunk,
    overlap
)

print("\nTaille des chunks :", taille_chunk)
print("Chevauchement :", overlap)
print("Nombre total de chunks :", len(chunks))

# Affichage d'un exemple
print("\nEXEMPLE DE CHUNK")
print("Source :", chunks[0]["source"])
print("Pages :", chunks[0]["page_debut"], "-", chunks[0]["page_fin"])
print("Contenu :", chunks[0]["texte"])

Nombre de PDF : 5
Nombre de pages extraites : 13

Taille des chunks : 400
Chevauchement : 50
Nombre total de chunks : 14

EXEMPLE DE CHUNK
Source : 01_guide_pfe.pdf
Pages : 1 - 1
Contenu : ISIT – Guide PFE – 2026-2027 – v3.0 Page 1 Guide du Projet de Fin d'Études (PFE) – ISIT – 2026-2027 Institut Supérieur d'Informatique et de Technologies (ISIT) · Version 3.0 · 2026-09-01 1. Définition et objectifs du PFE Le Projet de Fin d'Études (PFE) est l'épreuve finale du cycle ingénieur de l'ISIT, réalisée en troisième année. Il vaut 30 crédits ECTS et se déroule en entreprise ou en laboratoire de recherche. Son objectif est de mettre l'étudiant en situation professionnelle : analyser un besoin, concevoir, développer, tester et documenter une solution complète. Le PFE est obligatoire pour obtenir le diplôme d'ingénieur. 2. Conditions d'inscription au PFE Pour s'inscrire au PFE, l'étudiant doit avoir validé tous les modules des semestres 1 à 5 du cycle ingénieur, ou avoir au maximum un module non

## Étape 4 — Construction du système de recherche

### 4.1 Génération des embeddings


Afin de permettre la recherche sémantique, nous transformons chaque chunk en un vecteur numérique appelé embedding.

Nous utilisons le modèle multilingue Sentence Transformers :
`paraphrase-multilingual-MiniLM-L12-v2`.

Lorsqu'un étudiant pose une question, celle-ci est également transformée en embedding.

Le système compare ensuite les vecteurs à l'aide de la similarité cosinus et récupère les trois passages les plus pertinents (Top-3).


Chaque chunk est transformé en un vecteur numérique à l'aide d'un modèle Sentence Transformers multilingue. Ces vecteurs permettent de comparer le sens des textes et de retrouver les passages pertinents pour une question.

In [14]:
# Chargement du modèle d'embedding
MODELE_EMBEDDING = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_model = SentenceTransformer(MODELE_EMBEDDING)

# Génération des embeddings des chunks
textes_chunks = [chunk["texte"] for chunk in chunks]

embeddings = embedding_model.encode(
    textes_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Nombre de chunks :", len(chunks))
print("Dimensions des embeddings :", embeddings.shape)


# Fonction de recherche
def rechercher(question, liste_chunks, liste_embeddings, k=3):

    embedding_question = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = np.dot(
        liste_embeddings,
        embedding_question[0]
    )

    meilleurs_indices = np.argsort(scores)[::-1][:k]

    resultats = []

    for indice in meilleurs_indices:

        chunk = liste_chunks[indice]

        resultats.append({
            "score": float(scores[indice]),
            "texte": chunk["texte"],
            "source": chunk["source"],
            "page_debut": chunk["page_debut"],
            "page_fin": chunk["page_fin"]
        })

    return resultats

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Nombre de chunks : 14
Dimensions des embeddings : (14, 384)



## étape 5. Test de la recherche et amélioration du chunking

Nous testons d'abord la recherche documentaire avec les chunks initiaux de 400 mots.

Lors d'un premier essai, le passage contenant la durée du PFE n'apparaissait pas dans les trois premiers résultats, alors que cette information était bien présente dans le corpus.

Nous avons donc décidé de tester une taille de 200 mots avec un chevauchement de 40 mots afin d'améliorer la précision de la recherche.

Nous comparons les résultats obtenus avant et après cette modification.


In [15]:
# Question utilisée pour démontrer l'échec initial
question_test = "Quelle est la durée du Projet de Fin d'Études ?"

# Recherche avec les chunks initiaux (400 mots)
resultats_avant = rechercher(
    question_test,
    chunks,
    embeddings,
    k=3
)

print("AVANT CORRECTION : chunks de 400 mots")

for i, resultat in enumerate(resultats_avant, start=1):
    print("\nPassage", i)
    print("Score :", round(resultat["score"], 4))
    print("Source :", resultat["source"])
    print("Contenu :", resultat["texte"][:400])


# Création des nouveaux chunks (200 mots)
nouvelle_taille_chunk = 200
nouvel_overlap = 40

nouveaux_chunks = creer_chunks(
    pages_data,
    nouvelle_taille_chunk,
    nouvel_overlap
)

# Génération des nouveaux embeddings
nouveaux_textes = [
    chunk["texte"] for chunk in nouveaux_chunks
]

nouveaux_embeddings = embedding_model.encode(
    nouveaux_textes,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Recherche après correction
resultats_apres = rechercher(
    question_test,
    nouveaux_chunks,
    nouveaux_embeddings,
    k=3
)

print("\n" + "=" * 70)
print("APRÈS CORRECTION : chunks de 200 mots")
print("=" * 70)

for i, resultat in enumerate(resultats_apres, start=1):
    print("\nPassage", i)
    print("Score :", round(resultat["score"], 4))
    print("Source :", resultat["source"])
    print("Contenu :", resultat["texte"][:400])

AVANT CORRECTION : chunks de 400 mots

Passage 1
Score : 0.7104
Source : 01_guide_pfe.pdf
Contenu : validée par la commission des PFE avant le 4 décembre 2026. Une fiche incomplète est renvoyée à l'étudiant avec des remarques ; il dispose de 7 jours pour la corriger. 6. Encadrement : rôles et responsabilités Chaque étudiant a deux encadrants. • L'encadrant académique est un enseignant de l'ISIT, affecté au plus tard le 16 novembre 2026. Il valide la démarche scientifique, relit le rapport et org

Passage 2
Score : 0.6587
Source : 05_calendrier_universitaire.pdf
Contenu : ISIT – Calendrier universitaire – 2026-2027 – v1.4 Page 1 Calendrier universitaire – ISIT – 2026-2027 Institut Supérieur d'Informatique et de Technologies (ISIT) · Version 1.4 · 2026-09-01 1. Rentrée et organisation de l'année L'année universitaire 2026-2027 commence le lundi 14 septembre 2026 pour tous les niveaux. Elle comprend deux semestres : le semestre 1 (septembre à janvier) et le semestre 

Passage 3
Score : 0.

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


APRÈS CORRECTION : chunks de 200 mots

Passage 1
Score : 0.7195
Source : 01_guide_pfe.pdf
Contenu : 3. Durée et période du PFE Le PFE dure au minimum 20 semaines et au maximum 24 semaines, soit environ 5 à 6 mois. Pour l'année 2026-2027, il commence le 1er février 2027 et doit se terminer au plus tard le 11 juin 2027, date limite de dépôt du rapport final. Un temps plein de 35 à 40 heures par semaine est attendu. 4. Choix du sujet et de la structure d'accueil L'étudiant trouve lui-même sa struct

Passage 2
Score : 0.6821
Source : 01_guide_pfe.pdf
Contenu : titre, la problématique, les objectifs, les technologies envisagées, le planning prévisionnel et les coordonnées de l'encadrant professionnel. La fiche est signée par l'entreprise puis soumise au département. Elle doit être validée par la commission des PFE avant le 4 décembre 2026. Une fiche incomplète est renvoyée à l'étudiant avec des remarques ; il dispose de 7 jours pour la corriger. 6. Encad

Passage 3
Score : 0.6587
Source : 

## étape 5 - Ajouter la génération

### 5.1 Construction du prompt RAG


Génération des réponses avec le RAG

Après la recherche documentaire, nous intégrons le modèle Qwen 2.5 7B avec Ollama.

Le système transmet au modèle :
- La question de l'étudiant.
- Les trois passages documentaires les plus pertinents.
- Les noms des documents et les numéros de pages.

Un prompt spécifique impose au modèle de répondre uniquement à partir des informations fournies et de signaler lorsqu'une réponse est absente du corpus.

Le modèle génère ensuite une réponse accompagnée de ses sources.


In [16]:
# Construction du prompt
def construire_prompt(question, resultats):

    contexte = ""

    for i, resultat in enumerate(resultats, start=1):

        contexte += (
            f"\n[Passage {i}]\n"
            f"Document : {resultat['source']}\n"
            f"Pages : {resultat['page_debut']} - "
            f"{resultat['page_fin']}\n"
            f"Contenu : {resultat['texte']}\n"
        )

    prompt = f"""
Tu es PFE Navigator, un assistant spécialisé dans les
informations relatives au Projet de Fin d'Études de l'ISIT.

Ta mission est de répondre aux questions des étudiants.

RÈGLES OBLIGATOIRES :
1. Utilise exclusivement les informations du contexte.
2. N'invente aucune information.
3. Si le contexte ne contient pas la réponse, réponds exactement :
   "Je ne sais pas d'après les documents fournis."
4. Cite les documents et les pages utilisés.
5. Réponds clairement en français.

CONTEXTE DOCUMENTAIRE :
{contexte}

QUESTION DE L'ÉTUDIANT :
{question}

RÉPONSE :
"""

    return prompt


# Fonction de génération RAG avec Ollama
def repondre_avec_rag(question, k=3):

    resultats = rechercher(
        question,
        nouveaux_chunks,
        nouveaux_embeddings,
        k=k
    )

    prompt = construire_prompt(
        question,
        resultats
    )

    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.1,
            "num_predict": 500
        }
    )

    reponse = response.message.content

    return {
        "question": question,
        "reponse": reponse,
        "sources": [
            {
                "document": resultat["source"],
                "pages": (
                    f"{resultat['page_debut']} - "
                    f"{resultat['page_fin']}"
                )
            }
            for resultat in resultats
        ]
    }


# Test RAG sur les cinq questions
resultats_rag = []

for i, question in enumerate(questions, start=1):

    print("\n" + "=" * 70)
    print(f"QUESTION {i} : {question}")
    print("=" * 70)

    try:

        resultat = repondre_avec_rag(question, k=3)

        resultats_rag.append(resultat)

        print("\nRÉPONSE DE PFE NAVIGATOR :")
        print(resultat["reponse"])

        print("\nSOURCES UTILISÉES :")

        for source in resultat["sources"]:
            print(
                "-",
                source["document"],
                "| Pages :",
                source["pages"]
            )

    except Exception as e:

        print("Erreur lors de la génération :", str(e))

        resultats_rag.append({
            "question": question,
            "reponse": None,
            "sources": [],
            "erreur": str(e)
        })


# Sauvegarde des résultats RAG
with open(
    DOSSIER_RESULTATS / "resultats_rag.json",
    "w",
    encoding="utf-8"
) as fichier:

    json.dump(
        resultats_rag,
        fichier,
        ensure_ascii=False,
        indent=4
    )

print("\nTest RAG terminé.")


QUESTION 1 : Quelle est la date limite de dépôt du rapport final de PFE, et sous quelle forme faut-il le remettre ?

RÉPONSE DE PFE NAVIGATOR :
La date limite de dépôt du rapport final de PFE est le 11 juin 2027. Il doit être déposé sur la plateforme ISIT-Stages en PDF, avec un exemplaire relié remis au secrétariat du département.

Citation des documents utilisés :
- Document : 05_calendrier_universitaire.pdf, Page 2
- Document : 01_guide_pfe.pdf, Pages 2 - 2

SOURCES UTILISÉES :
- 05_calendrier_universitaire.pdf | Pages : 1 - 2
- 01_guide_pfe.pdf | Pages : 2 - 2
- 01_guide_pfe.pdf | Pages : 2 - 2

QUESTION 2 : Comment est calculée la note finale du PFE, et à partir de quelle note est-il validé ?

RÉPONSE DE PFE NAVIGATOR :
La note finale du PFE est calculée en moyennant les notes attribuées par chaque membre du jury, selon le barème mentionné dans le Guide de soutenance. Le PFE est validé avec une note supérieure ou égale à 10/20.

Ces informations proviennent du document 04_guide_so

### Comparaison finale : LLM seul vs RAG

In [17]:
print("=" * 100)
print("COMPARAISON : LLM SEUL VS RAG")
print("=" * 100)

for i in range(len(questions)):

    print(f"\nQUESTION {i+1} : {questions[i]}")

    print("\nLLM SEUL :")

    if i < len(resultats_llm):
        print(resultats_llm[i]["reponse"])
    else:
        print("Résultat indisponible.")

    print("\nRAG :")

    if i < len(resultats_rag):
        print(resultats_rag[i]["reponse"])
    else:
        print("Résultat indisponible.")

    print("\n" + "-" * 100)

COMPARAISON : LLM SEUL VS RAG

QUESTION 1 : Quelle est la date limite de dépôt du rapport final de PFE, et sous quelle forme faut-il le remettre ?

LLM SEUL :
Pour répondre précisément à votre question, il serait utile de connaître les détails spécifiques de votre programme de PFE (Projet Final d'Études) ou de votre institution d'enseignement. Cependant, je peux vous donner quelques conseils généraux sur la date limite et la forme de remise du rapport final :

1. Date limite de dépôt :
   - Elle est généralement fixée par votre université ou l'organisme responsable du PFE.
   - Elle peut varier selon les filières et les établissements.
   - Elle est souvent annoncée au début de l'année académique ou à la fin du semestre.

2. Forme de remise du rapport :
   - Version papier : Certains établissements préfèrent une version imprimée.
   - Version électronique : De plus en plus d'établissements acceptent des versions numériques.
   - Version PDF : C'est souvent la forme recommandée, car ell


## 7. Bilan et conclusion

La comparaison entre le LLM seul et le système RAG permet d'évaluer l'apport de la recherche documentaire.

Le LLM seul répond à partir de ses connaissances générales, sans accès direct aux documents de l'ISIT.

Le système RAG récupère des passages pertinents dans le corpus avant de générer une réponse. Il permet ainsi de mieux contextualiser les réponses et de fournir des références documentaires.

Cependant, la qualité du résultat dépend de la pertinence des passages récupérés et de la capacité du modèle à respecter le contexte fourni.

**Limites identifiées :**
- La recherche sémantique peut récupérer des passages non pertinents.
- Le modèle local peut produire des réponses incorrectes.
- Le temps de génération dépend des ressources matérielles.
- Une information absente du corpus doit être signalée plutôt qu'inventée.
